# Athlete Health RAG — findings & retrieval quality

A short analytical walk-through of what the context layer surfaces from the wearable
data, aimed at a non-engineering reader (the "contribute to presentations for senior
management" part of the role). Everything here reads from the same SQLite store the
RAG pipeline uses, so the charts and the chatbot answers are backed by identical data.

**Prerequisite:** run the pipeline once so the store exists:
```
python -m src.cli ingest data/athlete_a_wearable_daily.csv
python -m src.cli index
```

This notebook needs only `matplotlib` on top of the core project.

> This is an engineering demonstration over wearable *wellness* data and is **not**
> medical, diagnostic, or training advice.

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

ATHLETE = "athlete-a"
DB = Path("athlete_health.db")
assert DB.exists(), "Run the ingest + index CLI steps first (see the intro cell)."

conn = sqlite3.connect(str(DB))
conn.row_factory = sqlite3.Row
rows = conn.execute(
    """SELECT m.date, m.resting_hr, m.hrv_ms, m.hrv_baseline_low, m.hrv_baseline_high,
              m.sleep_hours, m.sleep_score, m.acwr, f.hrv_state, f.acwr_band,
              f.readiness_flag, f.sleep_debt_hours
         FROM daily_metrics m JOIN daily_features f
           ON m.athlete_id = f.athlete_id AND m.date = f.date
        WHERE m.athlete_id = ? ORDER BY m.date""",
    (ATHLETE,),
).fetchall()
conn.close()

dates = [datetime.strptime(r["date"], "%Y-%m-%d") for r in rows]
print(f"{len(rows)} days loaded: {rows[0]['date']} -> {rows[-1]['date']}")

## 1. Training load (ACWR) and the injury-risk bands

The acute:chronic workload ratio compares the last ~7 days of load to the last ~28.
The widely cited "sweet spot" is 0.8–1.3; above 1.5 is a high-risk zone. We shade the
bands so a reader sees at a glance when load drifted out of the safe range.

In [ ]:
acwr = [r["acwr"] for r in rows]

fig, ax = plt.subplots(figsize=(11, 4))
ax.axhspan(0.0, 0.8, color="#f0f0f0", label="detraining (<0.8)")
ax.axhspan(0.8, 1.3, color="#d9f0d9", label="optimal (0.8–1.3)")
ax.axhspan(1.3, 1.5, color="#fff0cc", label="caution (1.3–1.5)")
ax.axhspan(1.5, 2.2, color="#f7d6d6", label="high risk (>1.5)")
ax.plot(dates, acwr, color="#333", marker="o", ms=3, lw=1.3)
ax.set_title(f"Acute:Chronic Workload Ratio — {ATHLETE}")
ax.set_ylabel("ACWR")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.set_ylim(0, 1.7)
plt.tight_layout(); plt.show()

peak = max(rows, key=lambda r: r["acwr"] or 0)
print(f"Peak load day: {peak['date']} (ACWR {peak['acwr']}) — the single caution-zone day.")

**Takeaway.** This athlete stayed conservative. Load touched the caution band exactly
once (2026-09-03, ACWR 1.4, right after a 966 kcal run) and otherwise oscillated between
optimal and detraining as the block tapered down through October. No sustained danger.

## 2. HRV against the *personal* baseline band

A single HRV number means little; what matters is where it sits relative to the athlete's
own rolling baseline. Readings that dip below the baseline-low line flag suppressed
recovery. We highlight those days in red.

In [ ]:
hrv = [r["hrv_ms"] for r in rows]
lo = [r["hrv_baseline_low"] for r in rows]
hi = [r["hrv_baseline_high"] for r in rows]
sup_x = [d for d, r in zip(dates, rows) if r["hrv_state"] == "suppressed"]
sup_y = [r["hrv_ms"] for r in rows if r["hrv_state"] == "suppressed"]

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(dates, lo, hi, color="#d9e8ff", label="personal baseline band")
ax.plot(dates, hrv, color="#1f4e79", lw=1.3, marker="o", ms=3, label="morning HRV")
ax.scatter(sup_x, sup_y, color="#c0392b", zorder=5, s=40, label="suppressed")
ax.set_title(f"HRV vs personal baseline — {ATHLETE}")
ax.set_ylabel("ms")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

print("Suppressed-HRV days:", ", ".join(r["date"] for r in rows if r["hrv_state"] == "suppressed"))

**Takeaway.** Recovery held inside the baseline band for most of the window, then a
cluster of suppressed readings appeared in late September / early October — the body
signalling accumulated fatigue even as training load itself was easing. This is exactly
the kind of cross-signal pattern a single metric would miss.

## 3. Sleep — hours, score, and the worst nights

In [ ]:
sh = [r["sleep_hours"] for r in rows]
ss = [r["sleep_score"] for r in rows]

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
a1.bar(dates, sh, color="#4c72b0")
a1.axhline(8.0, color="#c0392b", ls="--", lw=1, label="8 h target")
a1.set_ylabel("sleep hours"); a1.legend(fontsize=8); a1.set_title(f"Sleep — {ATHLETE}")
a2.plot(dates, ss, color="#55a868", marker="o", ms=3)
a2.axhline(60, color="#e67e22", ls="--", lw=1, label="poor <60")
a2.set_ylabel("sleep score"); a2.legend(fontsize=8)
a2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.tight_layout(); plt.show()

worst = sorted((r for r in rows if r["sleep_score"]), key=lambda r: r["sleep_score"])[:5]
print("Worst nights:")
for r in worst:
    print(f"  {r['date']}  {r['sleep_hours']} h  score {r['sleep_score']}")

**Takeaway.** Sleep was the weakest pillar. Several nights crashed well below target —
the worst being 2026-10-19 (essentially no recorded sleep) and 2026-10-11 (2.1 h). These
poor nights line up with the suppressed-HRV cluster, reinforcing the under-recovery story.

## 4. Does retrieval actually find the right days?

The charts above are the ground truth. The RAG layer should retrieve *these same days*
when asked about them in plain language. The evaluation harness measures that with
standard IR metrics against a curated gold set.

In [ ]:
from src.evaluate import evaluate
from src.retriever import Retriever

report = evaluate(Retriever(), k=6)
print(f"embedder gives:  mean P@k={report.mean_precision}  mean R@k={report.mean_recall}  "
      f"hit-rate={report.hit_rate}  MRR={report.mrr}\n")
for s in report.per_query:
    print(f"  P={s.precision_at_k:.2f} R={s.recall_at_k:.2f} RR={s.reciprocal_rank:.2f}  {s.question}")

**Takeaway.** With the zero-dependency offline hashing embedder, every gold question
still surfaces a relevant day (hit-rate 1.0, MRR 1.0) thanks to the structured
metadata pre-filter; precision is moderate because the weak embedder can't rank-order
perfectly *within* a band. Installing `sentence-transformers` lifts precision with no
code change — the retrieval plumbing is identical across embedding backends. Measuring
this is what lets you iterate on context relevance deliberately instead of by vibe.